# 在 SageMaker 上正确部署 Embedding 模型

本 notebook 展示如何使用自定义 inference 脚本在 SageMaker 上部署 `BIaoo/lca-qwen3-embedding` 模型。


## 1. 环境准备

In [ ]:
!pip install -q sagemaker boto3 transformers torch  xtarfile[zstd]

In [ ]:
import sagemaker
import boto3
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.serverless import ServerlessInferenceConfig
import json
import tarfile
from pathlib import Path

# 获取执行角色和 session
role = sagemaker.get_execution_role()
sess = sagemaker.Session()
region = sess.boto_region_name
bucket = sess.default_bucket()

print(f"SageMaker role: {role}")
print(f"SageMaker bucket: {bucket}")
print(f"SageMaker region: {region}")

## 2. 创建自定义 Inference 脚本

这是最关键的部分!我们需要实现正确的 embedding 处理逻辑。

In [ ]:
# 创建代码目录
!mkdir -p code

In [ ]:
%%writefile code/inference.py
"""
自定义 inference 脚本用于 embedding 模型部署

实现功能:
1. Last-token pooling - 与 SentenceTransformer 配置一致
2. L2 normalization - 归一化向量
3. Attention mask 处理 - 正确处理 padding tokens
"""

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import json
import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)


def model_fn(model_dir):
    """
    加载模型和 tokenizer
    
    Args:
        model_dir: 模型文件目录
    
    Returns:
        tuple: (model, tokenizer)
    """
    logger.info(f"Loading model from {model_dir}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModel.from_pretrained(model_dir)
    
    # 设置为评估模式
    model.eval()
    
    # 如果有 GPU 可用,移动到 GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    logger.info(f"Model loaded successfully on {device}")
    
    return model, tokenizer


def last_token_pooling(token_embeddings, attention_mask):
    """
    选择每个序列的最后一个有效 token
    
    Args:
        token_embeddings: token 级别的 embeddings [batch_size, seq_len, hidden_size]
        attention_mask: attention mask [batch_size, seq_len]
    
    Returns:
        sentence_embeddings: 句子级别的 embeddings [batch_size, hidden_size]
    """
    # attention_mask.sum(dim=1) 给出有效 token 数量, 减 1 得到最后索引
    last_indices = attention_mask.sum(dim=1).clamp(min=1) - 1
    batch_indices = torch.arange(token_embeddings.size(0), device=token_embeddings.device)
    return token_embeddings[batch_indices, last_indices]


def predict_fn(data, model_and_tokenizer):
    """
    执行推理
    
    Args:
        data: 输入数据,格式:
            {
                "inputs": ["text1", "text2", ...] 或 "single text"
            }
        model_and_tokenizer: (model, tokenizer) tuple
    
    Returns:
        embeddings: 归一化后的 embeddings
    """
    model, tokenizer = model_and_tokenizer
    device = next(model.parameters()).device
    
    # 解析输入
    if isinstance(data, dict):
        texts = data.get('inputs', [])
    else:
        texts = data
    
    # 确保 texts 是列表
    if isinstance(texts, str):
        texts = [texts]
    
    logger.info(f"Processing {len(texts)} texts")
    
    # Tokenize
    encoded_input = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=1024,
        return_tensors='pt'
    )
    
    # 移动到正确的设备
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    
    # 推理
    with torch.no_grad():
        model_output = model(**encoded_input)
    
    # Last-token pooling (与训练配置一致)
    sentence_embeddings = last_token_pooling(
        model_output.last_hidden_state,
        encoded_input['attention_mask']
    )
    
    # L2 Normalization
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
    
    # 转换为列表
    embeddings = sentence_embeddings.cpu().numpy().tolist()
    
    logger.info(f"Generated embeddings with shape: {sentence_embeddings.shape}")
    
    return embeddings


def input_fn(request_body, content_type='application/json'):
    """
    解析输入数据
    
    Args:
        request_body: 请求体
        content_type: 内容类型
    
    Returns:
        解析后的数据
    """
    if content_type == 'application/json':
        return json.loads(request_body)
    else:
        raise ValueError(f"Unsupported content type: {content_type}")


def output_fn(prediction, accept='application/json'):
    """
    格式化输出
    
    Args:
        prediction: 预测结果
        accept: 接受的内容类型
    
    Returns:
        格式化后的响应
    """
    if accept == 'application/json':
        return json.dumps({
            'embeddings': prediction,
            'model': 'BIaoo/lca-qwen3-embedding',
            'normalized': True
        }), accept
    else:
        raise ValueError(f"Unsupported accept type: {accept}")


## 3. 创建 requirements.txt

In [ ]:
%%writefile code/requirements.txt
transformers==4.51.3
torch==2.6.0
sentencepiece

## 4. 打包模型和代码

我们需要将模型文件和 code/ 里的 inference.py、requirements.txt 一起打包成 model.tar.gz 上传到 S3。


In [ ]:
from transformers import AutoTokenizer, AutoModel

model_id = "BIaoo/lca-qwen3-embedding"
model_dir = "model"

print(f"Downloading model: {model_id}")

# 下载模型和 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

# 保存到本地
tokenizer.save_pretrained(model_dir)
model.save_pretrained(model_dir)

print(f"Model saved to {model_dir}")

In [ ]:
# 将自定义代码放到模型目录下的 code/ 子目录
!mkdir -p {model_dir}/code
!cp code/inference.py {model_dir}/code/
!cp code/requirements.txt {model_dir}/code/


In [ ]:
# 打包成 tar.gz (需要 .gz 格式 - SageMaker 要求)
import tarfile
import os
from pathlib import Path
import time

model_dir = "model"
model_archive = "model.tar.gz"

print(f"📦 开始打包: {model_dir} -> {model_archive}")
print(f"💡 提示: 打包过程较慢是因为 gzip 压缩(SageMaker 要求)\n")

# 计算文件数量和大小
files = list(Path(model_dir).rglob('*'))
file_count = len([f for f in files if f.is_file()])
total_size = sum(f.stat().st_size for f in files if f.is_file())

print(f"📊 统计信息:")
print(f"  - 文件数量: {file_count}")
print(f"  - 原始大小: {total_size / 1024 / 1024:.2f} MB")
print(f"\n⏳ 正在打包压缩...")

start_time = time.time()

# 打包
with tarfile.open(model_archive, "w:gz") as tar:
    tar.add(model_dir, arcname=".")

elapsed = time.time() - start_time
archive_size = os.path.getsize(model_archive)

print(f"\n✅ 打包完成!")
print(f"⏱️  耗时: {elapsed:.2f} 秒")
print(f"📦 压缩后: {archive_size / 1024 / 1024:.2f} MB")
print(f"📉 压缩率: {(1 - archive_size / total_size) * 100:.1f}%")
print(f"🚀 速度: {total_size / elapsed / 1024 / 1024:.2f} MB/s")

print(f"\n💡 性能优化建议:")
print(f"  - gzip 压缩较慢(CPU密集),但能减小上传大小")
print(f"  - 如果模型很大(>500MB),首次打包可能需要几分钟")
print(f"  - 后续重新打包会更快(文件系统缓存)")

## 5. 上传到 S3

In [ ]:
# 上传模型到 S3
s3_model_uri = sess.upload_data(
    path=model_archive,
    bucket=bucket,
    key_prefix="lca-embedding/model"
)

print(f"Model uploaded to: {s3_model_uri}")

## 6. 部署模型

### 选项 A: Serverless Inference (推荐)

适合间歇性使用,自动扩缩容。

In [ ]:
# 创建 HuggingFaceModel
s3_model_uri='s3://tiangong-models/lca-embedding-model.tar.gz'

huggingface_model = HuggingFaceModel(
    model_data=s3_model_uri,
    role=role,
    transformers_version='4.51.3',
    pytorch_version='2.6.0',
    py_version='py312',
    entry_point='inference.py'
)

# Serverless 配置
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=4096,  # 4GB 内存
    max_concurrency=10       # 最大并发数
)

# 部署
predictor = huggingface_model.deploy(
    serverless_inference_config=serverless_config
)

print(f"Model deployed to endpoint: {predictor.endpoint_name}")

### 选项 B: Real-time Inference

如果需要持续使用,可以部署实时端点。

In [ ]:
# 注释掉,如需使用请取消注释

# predictor = huggingface_model.deploy(
#     initial_instance_count=1,
#     instance_type='ml.g4dn.xlarge',  # GPU 实例
#     # instance_type='ml.m5.xlarge',  # 或 CPU 实例
# )
# 
# print(f"Model deployed to endpoint: {predictor.endpoint_name}")

## 7. 测试推理

In [ ]:
# 测试单个文本
from sagemaker.huggingface import HuggingFacePredictor
import sagemaker

# Endpoint 名称
endpoint_name = "huggingface-pytorch-inference-2026-01-13-03-50-14-488"

# 创建 predictor 对象连接到现有 endpoint
predictor = HuggingFacePredictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sagemaker.Session()
)

test_input = {
    "inputs": "This is a test sentence for embedding generation."
}

response = predictor.predict(test_input)
print(json.dumps(response, indent=2))

In [ ]:
# 测试多个文本
test_input_batch = {
    "inputs": [
        "wood residue gasification heat recovery",
        "steel production blast furnace",
        "solar panel manufacturing"
    ]
}

response = predictor.predict(test_input_batch)
print(f"Generated {len(response['embeddings'])} embeddings")
print(f"Embedding dimension: {len(response['embeddings'][0])}")

## 8. 验证输出正确性

关键检查点:
1. ✅ 向量已归一化 (范数应该 ≈ 1.0)
2. ✅ 向量维度正确 (1024)
3. ✅ 数值范围合理 (约 -0.1 到 0.1)

In [ ]:
import numpy as np

# 获取第一个 embedding
embedding = np.array(response['embeddings'][0])

# 计算范数
norm = np.linalg.norm(embedding)
print(f"Vector norm: {norm:.6f}")
print(f"Is normalized: {abs(norm - 1.0) < 0.01}")

# 统计信息
print(f"\nVector statistics:")
print(f"  Dimension: {len(embedding)}")
print(f"  Mean: {np.mean(embedding):.6f}")
print(f"  Std: {np.std(embedding):.6f}")
print(f"  Min: {np.min(embedding):.6f}")
print(f"  Max: {np.max(embedding):.6f}")

# 验证结果
assert abs(norm - 1.0) < 0.01, "❌ Vector is not normalized!"
assert len(embedding) == 1024, "❌ Incorrect embedding dimension!"
assert -0.2 < np.min(embedding) < 0.2, "❌ Value range seems incorrect!"

print("\n✅ All checks passed! SageMaker deployment is correct.")

## 9. 与其他平台对比

如果你有来自 vLLM/Ollama/AWS 的 embedding,可以计算相似度进行对比。

In [ ]:
def cosine_similarity(vec1, vec2):
    """计算余弦相似度"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 如果有参考向量,可以对比
# reference_embedding = np.array([...])  # 从 vLLM/Ollama/AWS 获取
# similarity = cosine_similarity(embedding, reference_embedding)
# print(f"Similarity with reference: {similarity:.6f}")
# 
# # 应该 >99.9%
# assert similarity > 0.999, f"❌ Similarity too low: {similarity}"
# print("✅ High similarity confirmed!")

## 10. 连接到现有端点(可选)

如果端点已存在,可以直接连接。

In [ ]:
from sagemaker.huggingface import HuggingFacePredictor

# 替换为你的端点名称
endpoint_name = "huggingface-pytorch-inference-2026-01-12-xx-xx-xx-xxx"

# 创建 predictor
predictor = HuggingFacePredictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sess
)

# 测试
response = predictor.predict({"inputs": "test"})
print(json.dumps(response, indent=2))

## 11. 清理资源

⚠️ 不使用时记得删除端点,避免产生费用!

In [ ]:
# 删除端点
# predictor.delete_endpoint()
# print("Endpoint deleted")

## 总结

### ✅ 关键要点

1. **必须使用自定义 inference.py**
   - SageMaker 不支持 embedding 任务类型
   - 需要手动实现 mean pooling + L2 归一化

2. **验证输出正确性**
   - 范数应该 ≈ 1.0
   - 与 vLLM/Ollama 的相似度应该 >99.9%

3. **成本考虑**
   - Serverless: 按实际调用次数付费,适合间歇性使用
   - Real-time: 按实例运行时间付费,适合持续使用

### 📚 相关文档

- [analysis/SAGEMAKER_ANALYSIS.md](analysis/SAGEMAKER_ANALYSIS.md) - 问题深度分析
- [analysis/FINAL_SUMMARY.md](analysis/FINAL_SUMMARY.md) - 完整分析报告
- [SageMaker Custom Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/adapt-inference-container.html)

### 💡 推荐

如果可能,建议使用 **vLLM** 或 **Ollama**,它们:
- ✅ 开箱即用,无需自定义代码
- ✅ 性能更好
- ✅ 部署更简单
- ✅ 成本更低

详见: [analysis/README.md](analysis/README.md)